In [4]:
!pip install mediapipe pandas tqdm --quiet

In [5]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import pandas as pd
import os
import pickle
import urllib.request
from tqdm import tqdm

PILOT_FRAMES_DIR = "D:/UGC_Project/pilot_frames"
FEATURES_DIR = "D:/UGC_Project/pilot_features"
os.makedirs(FEATURES_DIR, exist_ok=True)

LANDMARKS_DIR = os.path.join(FEATURES_DIR, "raw_landmarks")
os.makedirs(LANDMARKS_DIR, exist_ok=True)

# Download model files (one-time)
face_model_path = "face_landmarker.task"
pose_model_path = "pose_landmarker.task"

if not os.path.exists(face_model_path):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task",
        face_model_path)
    print("Face model downloaded")

if not os.path.exists(pose_model_path):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task",
        pose_model_path)
    print("Pose model downloaded")

BaseOptions = python.BaseOptions
VisionRunningMode = vision.RunningMode

face_landmarker = vision.FaceLandmarker.create_from_options(
    vision.FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=face_model_path),
        running_mode=VisionRunningMode.IMAGE,
        num_faces=1))

pose_landmarker = vision.PoseLandmarker.create_from_options(
    vision.PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=pose_model_path),
        running_mode=VisionRunningMode.IMAGE))

LEFT_EYE_TOP, LEFT_EYE_BOTTOM = 159, 145
RIGHT_EYE_TOP, RIGHT_EYE_BOTTOM = 386, 374
MOUTH_TOP, MOUTH_BOTTOM = 13, 14
NOSE_TIP = 1
LEFT_CHEEK, RIGHT_CHEEK = 234, 454

BLINK_THRESHOLD = 0.015

Face model downloaded
Pose model downloaded


In [6]:
def get_frame_features(frame):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    face_result = face_landmarker.detect(mp_image)
    pose_result = pose_landmarker.detect(mp_image)

    features = {
        "face_detected": 0, "pose_detected": 0,
        "eye_openness": None, "mouth_open": None, "head_yaw": None,
        "shoulder_diff": None, "wrist_l_x": None, "wrist_l_y": None,
        "wrist_r_x": None, "wrist_r_y": None,
    }
    face_landmarks_raw = None
    pose_landmarks_raw = None

    if face_result.face_landmarks:
        lm = face_result.face_landmarks[0]
        features["face_detected"] = 1
        left_eye = abs(lm[LEFT_EYE_TOP].y - lm[LEFT_EYE_BOTTOM].y)
        right_eye = abs(lm[RIGHT_EYE_TOP].y - lm[RIGHT_EYE_BOTTOM].y)
        features["eye_openness"] = (left_eye + right_eye) / 2
        features["mouth_open"] = abs(lm[MOUTH_TOP].y - lm[MOUTH_BOTTOM].y)
        features["head_yaw"] = abs(lm[NOSE_TIP].x - lm[LEFT_CHEEK].x) - abs(lm[NOSE_TIP].x - lm[RIGHT_CHEEK].x)
        face_landmarks_raw = [(p.x, p.y, p.z) for p in lm]

    if pose_result.pose_landmarks:
        lm = pose_result.pose_landmarks[0]
        features["pose_detected"] = 1
        features["shoulder_diff"] = abs(lm[11].y - lm[12].y)
        features["wrist_l_x"], features["wrist_l_y"] = lm[15].x, lm[15].y
        features["wrist_r_x"], features["wrist_r_y"] = lm[16].x, lm[16].y
        pose_landmarks_raw = [(p.x, p.y, p.z) for p in lm]

    return features, face_landmarks_raw, pose_landmarks_raw

In [7]:
def process_video(video_id, frames_dir):
    frame_files = sorted(os.listdir(frames_dir))
    per_frame_features = []
    landmark_sequence = []

    for fname in frame_files:
        frame_path = os.path.join(frames_dir, fname)
        frame = cv2.imread(frame_path)
        if frame is None:
            continue
        feats, face_lm, pose_lm = get_frame_features(frame)
        per_frame_features.append(feats)
        landmark_sequence.append({"face": face_lm, "pose": pose_lm})

    if not per_frame_features:
        return None

    df = pd.DataFrame(per_frame_features)
    total_frames = len(df)

    row = {
        "video_id": video_id,
        "total_frames": total_frames,
        "face_detection_rate": df["face_detected"].mean(),
        "pose_detection_rate": df["pose_detected"].mean(),
        "eye_openness_mean": df["eye_openness"].mean(),
        "eye_openness_std": df["eye_openness"].std(),
        "blink_rate": (df["eye_openness"] < BLINK_THRESHOLD).sum() / max(total_frames, 1),
        "mouth_open_mean": df["mouth_open"].mean(),
        "mouth_open_std": df["mouth_open"].std(),
        "head_yaw_mean": df["head_yaw"].mean(),
        "head_yaw_std": df["head_yaw"].std(),
        "shoulder_diff_mean": df["shoulder_diff"].mean(),
        "shoulder_diff_std": df["shoulder_diff"].std(),
    }

    wrist_l = df[["wrist_l_x", "wrist_l_y"]].dropna().values
    wrist_r = df[["wrist_r_x", "wrist_r_y"]].dropna().values
    row["wrist_l_motion"] = np.mean(np.linalg.norm(np.diff(wrist_l, axis=0), axis=1)) if len(wrist_l) > 1 else 0
    row["wrist_r_motion"] = np.mean(np.linalg.norm(np.diff(wrist_r, axis=0), axis=1)) if len(wrist_r) > 1 else 0

    with open(os.path.join(LANDMARKS_DIR, f"{video_id}.pkl"), "wb") as f:
        pickle.dump(landmark_sequence, f)

    return row

In [8]:
video_folders = os.listdir(PILOT_FRAMES_DIR)
print(f"Total videos to process: {len(video_folders)}")

results = []
for video_id in tqdm(video_folders, desc="Extracting features"):
    frames_dir = os.path.join(PILOT_FRAMES_DIR, video_id)
    row = process_video(video_id, frames_dir)
    if row is not None:
        results.append(row)

feature_df = pd.DataFrame(results)
feature_df.to_csv(os.path.join(FEATURES_DIR, "pilot_statistical_features.csv"), index=False)

print(f"\nProcessed videos: {len(feature_df)}")
print(f"Statistical features saved to: {os.path.join(FEATURES_DIR, 'pilot_statistical_features.csv')}")
print(f"Raw landmark sequences saved to: {LANDMARKS_DIR}")
print(f"\nAverage face detection rate: {feature_df['face_detection_rate'].mean():.2%}")
print(f"Average pose detection rate:   {feature_df['pose_detection_rate'].mean():.2%}")

Total videos to process: 100


Extracting features: 100%|███████████████████████████████████████████████████████████| 100/100 [40:51<00:00, 24.52s/it]


Processed videos: 98
Statistical features saved to: D:/UGC_Project/pilot_features\pilot_statistical_features.csv
Raw landmark sequences saved to: D:/UGC_Project/pilot_features\raw_landmarks

Average face detection rate: 30.33%
Average pose detection rate: 61.07%


In [9]:
import pandas as pd

df = pd.read_csv("D:/UGC_Project/pilot_features/pilot_statistical_features.csv")

print(df[["video_id", "face_detection_rate", "pose_detection_rate"]].sort_values("face_detection_rate"))
print(f"\nVideos with face_detection_rate < 20%: {(df['face_detection_rate'] < 0.2).sum()}")
print(f"Videos with face_detection_rate > 60%: {(df['face_detection_rate'] > 0.6).sum()}")

                        video_id  face_detection_rate  pose_detection_rate
48          pixabay_global_18784                  0.0             0.277778
57          pixabay_global_27095                  0.0             0.983871
56          pixabay_global_27023                  0.0             0.128415
55          pixabay_global_26793                  0.0             1.000000
54          pixabay_global_26296                  0.0             0.545455
..                           ...                  ...                  ...
74   SUGAR Cosmetics_9yXM-4Bqyy0                  1.0             0.983871
40         pixabay_global_129421                  1.0             1.000000
11  Dushyant kukreja_i2xtm_rlCeM                  1.0             1.000000
62          pixabay_global_61292                  1.0             0.549223
73   SUGAR Cosmetics_8sXVWiuBi-c                  1.0             1.000000

[98 rows x 3 columns]

Videos with face_detection_rate < 20%: 51
Videos with face_detection_rate > 